In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import torch
import numpy as np
from glob import glob
import os
import seaborn as sns

In [ ]:
paths = {
    # "BTCV+FIFO":"output/btcv+icl+no_agent+long_horizon+no_augment/2026-04-30-00-40-30/btcv__agentFalse_ablation.csv",
    # "BTCV+AFS":"output/btcv+icl+grpo+long_horizon+no_augment/2026-05-01-15-28-59/btcv__agentTrue_ablation.csv",
    "Sarcoma+FIFO": "output/sarcoma+icl+no_agent+long_horizon+no_augment/2026-04-10-19-47-31/sarcoma__agentFalse_ablation.csv",
    "Sarcoma+SAM2RL":"output/sarcoma+ppo+sam2rl/2026-05-06-23-41-48/sarcoma__agentTrue_ablation.csv",
    "Sarcoma+AFS": "output/sarcoma+icl+grpo+long_horizon+no_augment/2026-04-29-14-22-57/sarcoma_''_agentTrue_ablation.csv",
    "MSD-Heart+FIFO":"output/msd_task02+icl+no_agent+long_horizon+no_augment/2026-04-15-19-42-29/msd_Task02_agentFalse_ablation.csv",
    "MSD-Heart+SAM2RL":"output/msd_task02+ppo+sam2rl/2026-05-07-05-29-23/msd_Task02_agentTrue_ablation.csv",
    "MSD-Heart+AFS":"output/msd_task02+icl+grpo+lazy_pen0.1+long_horizon+flip_augment/2026-04-23-09-52-39/msd_Task02_agentTrue_ablation.csv",
    # "MSD-Heart+AFS":"output/msd_task02+icl+grpo+long_horizon+no_augment+lr2e-4/2026-05-01-07-12-31/msd_Task02_agentTrue_ablation.csv",
    # "MSD-Hippocampus+FIFO":"output/msd_task04+no_agent+icl+long_horizon+no_augment/2026-04-28-20-05-44/msd_Task04_agentFalse_ablation.csv",
    # "MSD-Hippocampus+AFS":"output/msd_task04+grpo+icl+long_horizon+no_augment/2026-04-27-11-50-03/msd_Task04_agentTrue_ablation.csv",
    "MSD-Prostate+FIFO":"output/msd_task05+no_agent+long_horizon+no_augment/2026-04-13-18-04-27/msd_Task05_agentFalse_ablation.csv",
    "MSD-Prostate+SAM2RL":"output/msd_task05+ppo+sam2rl/2026-05-06-23-41-28/msd_Task05_agentTrue_ablation.csv",
    "MSD-Prostate+AFS":"output/msd_task05+grpo+long_horizon+no_augment/2026-04-26-11-35-38/msd_Task05_agentTrue_ablation.csv",
    "MSD-Lung+FIFO":"output/msd_task06+icl+no_agent+long_horizon+no_augment/2026-04-28-17-12-14/msd_Task06_agentFalse_ablation.csv",
    "MSD-Lung+SAM2RL":"output/msd_task06+ppo+sam2rl/2026-05-06-19-23-41/msd_Task06_agentTrue_ablation.csv",
    "MSD-Lung+AFS":"output/msd_task06+grpo+long_horizon+no_augment/2026-04-26-16-12-37/msd_Task06_agentTrue_ablation.csv",
    # "MSD-Pancreas+FIFO":"output/msd_task07+no_agent+icl+long_horizon+no_augment/2026-04-28-20-07-36/msd_Task07_agentFalse_ablation.csv",
    # "MSD-Pancreas+AFS":"output/msd_task07+grpo+icl+long_horizon+no_augment/2026-04-26-17-18-17/msd_Task07_agentTrue_ablation.csv",
    # "MSD-HepaticVessel+FIFO":"output/msd_task08+no_agent+icl+long_horizon+no_augment/2026-04-28-20-20-31/msd_Task08_agentFalse_ablation.csv",
    # "MSD-HepaticVessel+AFS":"output/msd_task08+grpo+icl+long_horizon+no_augment/2026-04-26-17-25-02/msd_Task08_agentTrue_ablation.csv",
    "MSD-Spleen+FIFO": "output/msd_task09+icl+no_agent+long_horizon+no_augment/2026-04-11-12-19-00/msd_Task09_agentFalse_ablation.csv",
    "MSD-Spleen+SAM2RL":"output/msd_task09+ppo+sam2rl/2026-05-07-13-48-40/msd_Task09_agentTrue_ablation.csv",
    "MSD-Spleen+AFS":"output/msd_task09+icl+grpo+long_horizon+no_augment/2026-05-01-22-59-45/msd_Task09_agentTrue_ablation.csv",
    "MSD-Colon+FIFO":"output/msd_task10+icl+no_agent+long_horizon+no_augment/2026-04-12-15-52-38/msd_Task10_agentFalse_ablation.csv",
    "MSD-Colon+SAM2RL":"output/msd_task10+ppo+sam2rl/2026-05-06-18-53-29/msd_Task10_agentTrue_ablation.csv",
    "MSD-Colon+AFS":"output/msd_task10+icl+grpo+long_horizon+no_augment/2026-04-25-13-09-26/msd_Task10_agentTrue_ablation.csv",
}

In [ ]:
dfs = {}

for l, p in paths.items():
    print(l)
    df = pd.read_csv(p, index_col=0)
    dfs[l] = df

In [ ]:
for l, df in dfs.items():
    df["vol_id"] = df["id"].str.split("_").str[1].str.split(".").str[0].astype(np.int64)
    df["obj_id"] = df["id"].str.split("_").str[-2].astype(np.int64)
    df["frame_id"] = df["id"].str.split("_").str[-1].astype(np.int64)
    dfs[l] = df.set_index(["vol_id", "obj_id", "frame_id"])
    # dfs[l] = df.sort_values(["vol_id", "obj_id", "frame_id"])

In [ ]:
for label, df in dfs.items():
    drop_delta = []
    total, miss = 0, 0
    for frame_id, (d, f) in zip(df['id'], zip(df['delta'], df['drop_frame'])):
        if f == -1:
            v = 0
        else:
            dices = eval(d)
            deltas = []
            for i, dice in dices.items():
                if i == f:
                    continue
                deltas.append(dices[f] - dice)  # actual choice - other choices => higher means better  

            # v = np.min(deltas)
            v = dices[f] - np.mean(list(dices.values()))
        
        total += 1
        miss += (v > 0)
        drop_delta.append(v * 100)
        
    # print(label, miss / total)

    df['drop_delta'] = pd.Series(drop_delta, index=df.index)

In [ ]:
import functools
join_dfs = {}
mean_delta = {}
min_attn = {}
for dataset in np.unique([k.split("+")[0] for k in dfs.keys()]):
    rl_df = dfs[dataset+"+AFS"]
    sam2rl_df = dfs[dataset+"+SAM2RL"].add_suffix('_sam2rl')
    fifo_df = dfs[dataset+"+FIFO"]

    # display(sam2rl_df)
    join_df = rl_df.join(fifo_df, lsuffix="_rl", rsuffix="_fifo")
    join_df = join_df.join(sam2rl_df, lsuffix="", rsuffix="_sam2rl")
    # display(join_df)
    join_df['rl_fifo_dice_diff'] = (join_df['dice_rl'] - join_df['dice_fifo']) * 100
    join_df['rl_sam2rl_dice_diff'] = (join_df['dice_rl'] - join_df['dice_sam2rl']) * 100
    join_df['dice_diff'] = join_df[["rl_fifo_dice_diff", "rl_sam2rl_dice_diff"]].min(axis=1)
    join_df['attn_diff'] = join_df['min_attn_rl'] - join_df['min_attn_fifo']
    join_df = join_df[join_df["drop_frame_fifo"] != -1]
    # join_df = join_df[join_df['dice_diff'] > 0.0]
    # join_df = join_df[(join_df['dice_diff'] > join_df['dice_diff'].quantile(0.9))]
    join_df = join_df[(join_df['drop_delta_rl'] > join_df['drop_delta_rl'].quantile(0.6))]
    
    print(dataset, len(join_df))
    join_dfs[dataset] = join_df
    mean_delta[dataset+"+FIFO"] = join_df["drop_delta_fifo"]
    mean_delta[dataset+"+SAM2RL"] = join_df["drop_delta_sam2rl"]
    mean_delta[dataset+"+AFS"] = join_df["drop_delta_rl"]
    min_attn[dataset+"+FIFO"] = join_df["min_attn_fifo"]
    min_attn[dataset+"+SAM2RL"] = join_df["min_attn_sam2rl"]
    min_attn[dataset+"+AFS"] = join_df["min_attn_rl"]

join_dfs["Overall"] = pd.concat(join_dfs.values())
mean_delta["Overall+FIFO"] = join_dfs["Overall"]["drop_delta_fifo"]
mean_delta["Overall+SAM2RL"] = join_dfs["Overall"]["drop_delta_sam2rl"]
mean_delta["Overall+AFS"] = join_dfs["Overall"]["drop_delta_rl"]
# display(dfs["MSD-Heart+AFS"][dfs["MSD-Heart+AFS"]["id"] == "la_007.nii.gz_1_58"])

In [ ]:
# eval(join_dfs["Sarcoma"].loc[pd.MultiIndex.from_tuples([(14,2,18)], names=["vol_id", "obj_id", "frame_id"])]["delta_rl"].values[0])

In [ ]:
# eval(join_dfs["Sarcoma"].loc[pd.MultiIndex.from_tuples([(14,2,18)], names=["vol_id", "obj_id", "frame_id"])]["delta_sam2rl"].values[0])

In [ ]:
# join_dfs["Sarcoma"].loc[pd.MultiIndex.from_tuples([(14,2,18)], names=["vol_id", "obj_id", "frame_id"])]["drop_frame_rl"]

In [ ]:
# join_dfs["Sarcoma"].loc[pd.MultiIndex.from_tuples([(14,2,18)], names=["vol_id", "obj_id", "frame_id"])]["drop_frame_sam2rl"]

In [ ]:
display_cols = [
        "dice_diff",
        "rl_fifo_dice_diff",
        "rl_sam2rl_dice_diff"
        # "drop_delta_rl",
        # "drop_delta_fifo",
        # 'attn_diff',
        # "min_attn_rl",
        # "min_attn_fifo",
        # "delta_rl",
        # "delta_fifo",
        # "delta_sam2rl"
]

# with pd.option_context('display.max_rows', None, 'display.max_columns', None):
#         display(join_dfs["MSD-Spleen"].sort_values('dice_diff', ascending=False)[display_cols])

In [ ]:
# Set publication-quality style
sns.set_theme(style="whitegrid", font="DejaVu Sans", font_scale=1.2)
colors = ["#4C72B0", "#C44E52"]
sns.set_palette(sns.color_palette(colors))
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 10,
    "axes.labelsize": 10,
    "axes.titlesize": 12,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "figure.dpi": 300
})

# Example data structure
datasets = [
    # 'BTCV',
    'Sarcoma',
    'MSD-Heart',
    # 'MSD-Hippocampus',
    'MSD-Prostate',
    'MSD-Lung',
    # 'MSD-Pancreas',
    # 'MSD-HepaticVessel',
    # 'MSD-Spleen',
    'MSD-Colon',
    'Overall'
]
fifo_vals = [mean_delta[d + "+FIFO"].median() for d in datasets]
sam2rl_vals = [mean_delta[d + "+SAM2RL"].median() for d in datasets]
afs_vals = [mean_delta[d + "+AFS"].median() for d in datasets]
print(fifo_vals)
print(sam2rl_vals)
print(afs_vals)
# print(fifo)

# Plotting
fig, ax = plt.subplots(figsize=(10, 4))
width = 0.2
x = range(len(datasets))

ax.bar([i - width for i in x], fifo_vals, width, label='FIFO', color='r')
ax.bar([i for i in x], sam2rl_vals, width, label='SAM2RL', color='g')
ax.bar([i + width for i in x], afs_vals, width, label='AFS (Ours)', color='b')
ax.set_ylabel(r'$\Delta$Dice') # LaTeX formatting
ax.set_xticks(x)
ax.set_xticklabels(datasets, rotation=0)
ax.legend(frameon=True, loc='upper left')
ax.set_title(r'$\Delta$Dice')
sns.despine() # Removes unnecessary top/right borders
plt.tight_layout()
plt.savefig("ablation_quantitative_with_sam2rl.pdf")

In [ ]:
# Set publication-quality style
sns.set_theme(style="whitegrid", font="DejaVu Sans", font_scale=1.2)
colors = ["#4C72B0", "#C44E52"]
sns.set_palette(sns.color_palette(colors))
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 10,
    "axes.labelsize": 10,
    "axes.titlesize": 12,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "figure.dpi": 300
})

# Example data structure
datasets = [
    # 'BTCV',
    # 'Sarcoma',
    # 'MSD-Heart',
    # 'MSD-Hippocampus',
    # 'MSD-Prostate',
    # 'MSD-Lung',
    # 'MSD-Pancreas',
    # 'MSD-HepaticVessel',
    # 'MSD-Spleen',
    'MSD-Colon',
    # 'Overall'
]
fifo_vals = [(mean_delta[d + "+FIFO"] < 0.0).mean() for d in datasets]
afs_vals = [(mean_delta[d + "+AFS"] < 0.0).mean() for d in datasets]
print(fifo_vals)
print(afs_vals)
# print(fifo)

# Plotting
fig, ax = plt.subplots(figsize=(10, 5))
width = 0.35
x = range(len(datasets))

ax.bar([i - width/2 for i in x], fifo_vals, width, label='FIFO', color='r')
ax.bar([i + width/2 for i in x], afs_vals, width, label='AFS (Ours)', color='b')
ax.set_ylabel(r'Miss Drop Rate') # LaTeX formatting
ax.set_xticks(x)
ax.set_xticklabels(datasets, rotation=20)
ax.legend(frameon=True, loc='upper left')
ax.set_title(r'Miss Drop Rate')
sns.despine() # Removes unnecessary top/right borders
plt.tight_layout()
plt.savefig("ablation_quantitative.pdf")

In [ ]:
# Set publication-quality style
sns.set_theme(style="whitegrid", font="DejaVu Sans", font_scale=1.2)
colors = ["#4C72B0", "#C44E52"]
sns.set_palette(sns.color_palette(colors))
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 10,
    "axes.labelsize": 10,
    "axes.titlesize": 12,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "figure.dpi": 300
})

# Example data structure
datasets = [
    # 'BTCV',
    # 'Sarcoma',
    # 'MSD-Heart',
    # 'MSD-Hippocampus',
    # 'MSD-Prostate',
    # 'MSD-Lung',
    # 'MSD-Pancreas',
    # 'MSD-HepaticVessel',
    # 'MSD-Spleen',
    'MSD-Colon'
]
fifo_vals = [min_attn[d + "+FIFO"].median() for d in datasets]
afs_vals = [min_attn[d + "+AFS"].median() for d in datasets]


# Plotting
fig, ax = plt.subplots(figsize=(10, 6))
width = 0.35
x = range(len(datasets))

ax.bar([i - width/2 for i in x], fifo_vals, width, label='FIFO', color='r')
ax.bar([i + width/2 for i in x], afs_vals, width, label='AFS', color='b')
ax.set_ylabel('Min Attn') # LaTeX formatting
ax.set_xticks(x)
ax.set_xticklabels(datasets, rotation=45)
ax.legend(frameon=True, loc='upper left')
ax.set_title("Min attention")
sns.despine() # Removes unnecessary top/right borders
plt.tight_layout()
# plt.savefig("output/quantitative.pdf")

In [ ]:
# Set publication-quality style
sns.set_theme(style="whitegrid", font="DejaVu Sans", font_scale=1.2)
colors = ["#C44E52", "#4C72B0"]
sns.set_palette(sns.color_palette(colors))
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 10,
    "axes.labelsize": 10,
    "axes.titlesize": 12,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "figure.dpi": 300
})

# 1. Setup synthetic data (simulating distributions around your previous means)
np.random.seed(42)
datasets = [
    # 'Sarcoma',
    # 'MSD-Heart',
    # 'MSD-Hippocampus',
    # 'MSD-Prostate',
    # 'MSD-Lung',
    # 'MSD-Pancreas',
    # 'MSD-HepaticVessel',
    # 'MSD-Spleen',
    'MSD-Colon'
]

data = []
# Means from your previous chart
fifo_means = [0.85, 2.9, 2.1, 1.45]
afs_means = [-0.1, 1.35, 0.3, 1.38]

for ds in datasets:
    # Generating distribution data
    data.extend([{'Dataset': ds, 'Method': 'FIFO', 'DeltaDice': v} for v in mean_delta[ds+"+FIFO"]])
    data.extend([{'Dataset': ds, 'Method': 'AFS', 'DeltaDice': v} for v in mean_delta[ds+"+AFS"]])

df = pd.DataFrame(data)

# 3. Create the Boxplot
fig, ax = plt.subplots(figsize=(10, 6))

sns.boxplot(
    data=df,
    x='Dataset',
    y='DeltaDice',
    hue='Method',
    palette=colors,
    flierprops={'marker': 'o', 'markersize': 2, 'alpha': 0.3},
    showfliers=False, # Show outliers
    linewidth=0.8,
    ax=ax
)

# Professional tweaks
ax.axhline(0, color='black', linestyle='-', linewidth=1, zorder=0)
ax.set_ylabel(r'$\Delta$Dice')
ax.set_xlabel('') # Often dataset names on x-axis are self-explanatory
sns.despine(left=True) # Clean look
ax.grid(axis='y', linestyle=':', alpha=0.5, linewidth=0.5)
ax.set_axisbelow(True) # Ensure grid is behind the boxes
plt.legend(frameon=False, loc="best")
plt.tight_layout()

# Save as vector for final paper
# plt.savefig('output/quantitative-box.pdf', bbox_inches='tight')
# plt.show()